# Phase 6 — Model Training (PySpark MLlib)

**Project:** Flight Delay Prediction  
**Input:** `D:/project/data/processed/features_final.parquet`  
**Engine:** Apache Spark MLlib  

### Models Trained
| Model | Purpose |
|---|---|
| Logistic Regression | Baseline — interpretable, fast |
| Random Forest | Handles non-linearity, robust to nulls |
| Gradient Boosted Trees (GBT) | Best performer — sequential boosting |

### Train/Test Split
- **Temporal split** — train 2015–2021, test 2022–2024
- Avoids data leakage — future data never influences past training
- 2020 stays in training, isolated via `is_covid_year` flag

---
## 0. Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, OneHotEncoder,
    StandardScaler, Imputer
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
import pandas as pd
import os
os.environ["HADOOP_HOME"] = "D:/hadoop"
os.environ["PATH"] += ";D:/hadoop/bin"
os.environ["PYSPARK_PYTHON"] = "python"

PARQUET_IN = "D:/project/data/processed/features_final.parquet"
MODELS_DIR = "D:/project/models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Imports ready.")

Imports ready.


---
## 1. Start Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("FlightDelayModeling")
    .master("local[*]")
    .config("spark.driver.memory", "10g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready — UI at http://localhost:4040")

KeyboardInterrupt: 

---
## 2. Load Features

In [ ]:
df = spark.read.parquet(PARQUET_IN)

print(f"Total rows : {df.count():,}")
print(f"Columns    : {len(df.columns)}")

# Class balance
total   = df.count()
delayed = df.filter(F.col("DEP_DEL15") == 1).count()
on_time = total - delayed

delay_ratio  = delayed / total
weight_ratio = on_time / delayed

print(f"On-time  (0) : {on_time:,}  ({on_time/total*100:.1f}%)")
print(f"Delayed  (1) : {delayed:,}  ({delayed/total*100:.1f}%)")
print(f"Weight ratio : {weight_ratio:.2f}x  (upweight minority delayed class)")

---
## 3. Temporal Train / Test Split

**Train:** 2015–2021 | **Test:** 2022–2024  
Temporal split preserves time ordering — no future leakage into training.

In [ ]:
df = df.withColumn("FL_DATE", F.to_date("FL_DATE"))

train_df = df.filter(F.year("FL_DATE") <= 2021)
test_df  = df.filter(F.year("FL_DATE") >= 2022)

print(f"Train : {train_df.count():,} rows (2015-2021, includes 2020 flagged)")
print(f"Test  : {test_df.count():,} rows (2022-2024)")

train_df.cache()
test_df.cache()
print("Datasets cached.")

---
## 4. Feature Preparation for MLlib

MLlib requires all features in a single vector column.  
Pipeline: StringIndex → OneHotEncode → Impute nulls → Assemble → Scale

In [ ]:
CATEGORICAL_COLS = [
    "AIRPORT_CODE", "Dest", "Reporting_Airline",
    "season", "time_of_day", "wind_category",
    "visibility_category", "precip_category", "distance_bucket"
]

NUMERIC_COLS = [
    "month", "day_of_week", "dep_hour", "quarter",
    "is_weekend", "is_holiday_period", "week_of_year",
    "is_covid_year", "flight_volume_index",
    "avg_temp_c", "min_temp_c", "max_temp_c",
    "avg_visibility_km", "min_visibility_km",
    "avg_wind_ms", "max_wind_ms",
    "total_precip_mm", "avg_pressure_hpa",
    "avg_ceiling_m", "min_ceiling_m",
    "is_freezing", "is_extreme_heat", "temp_range_c",
    "is_high_wind", "is_low_visibility",
    "is_precipitation", "is_low_ceiling",
    "weather_severity_score",
    "is_hub", "airport_hist_delay_rate",
    "daily_flight_count", "congestion_index",
    "Distance", "route_total_flights",
    "carrier_hist_delay_rate",
]

LABEL_COL = "label"

# Cast label, add class weights
train_df = train_df.withColumn(LABEL_COL, F.col("DEP_DEL15").cast("double"))
test_df  = test_df.withColumn(LABEL_COL,  F.col("DEP_DEL15").cast("double"))
train_df = train_df.withColumn("classWeight",
    F.when(F.col(LABEL_COL) == 1.0, weight_ratio).otherwise(1.0))
test_df  = test_df.withColumn("classWeight", F.lit(1.0))

print(f"Categorical : {len(CATEGORICAL_COLS)} | Numeric : {len(NUMERIC_COLS)}")

In [ ]:
# Build preprocessing pipeline
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in CATEGORICAL_COLS
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in CATEGORICAL_COLS
]
imputer = Imputer(
    inputCols=NUMERIC_COLS,
    outputCols=[f"{c}_imp" for c in NUMERIC_COLS],
    strategy="median"
)
assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in CATEGORICAL_COLS] + [f"{c}_imp" for c in NUMERIC_COLS],
    outputCol="features_raw",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features",
    withMean=False, withStd=True
)

preprocessing = Pipeline(stages=indexers + encoders + [imputer, assembler, scaler])

print("Fitting preprocessing pipeline on training data...")
preprocessor = preprocessing.fit(train_df)
train_prep   = preprocessor.transform(train_df).cache()
test_prep    = preprocessor.transform(test_df).cache()
print("Preprocessing done.")

---
## 5. Evaluation Helper

In [ ]:
def evaluate_model(predictions, model_name):
    auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL, rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="f1"
    ).evaluate(predictions)

    precision = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedPrecision"
    ).evaluate(predictions)

    recall = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedRecall"
    ).evaluate(predictions)

    cm = (predictions
          .groupBy(LABEL_COL, "prediction")
          .count()
          .orderBy(LABEL_COL, "prediction")
          .toPandas())

    def get_cm(a, p):
        row = cm[(cm[LABEL_COL] == a) & (cm["prediction"] == p)]
        return int(row["count"].values[0]) if len(row) > 0 else 0

    tn, fp = get_cm(0.0, 0.0), get_cm(0.0, 1.0)
    fn, tp = get_cm(1.0, 0.0), get_cm(1.0, 1.0)

    print(f"\n{'='*52}")
    print(f"  {model_name}")
    print(f"{'='*52}")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"\n  Confusion Matrix (Test Set 2022-2024):")
    print(f"  {'':22s}  Pred 0       Pred 1")
    print(f"  {'Actual 0 (On-time)':<22s}  {tn:>10,}   {fp:>10,}")
    print(f"  {'Actual 1 (Delayed)':<22s}  {fn:>10,}   {tp:>10,}")
    print(f"{'='*52}")

    return {"model": model_name, "auc": round(auc,4), "f1": round(f1,4),
            "precision": round(precision,4), "recall": round(recall,4),
            "TP": tp, "TN": tn, "FP": fp, "FN": fn}

results = []
print("Evaluator ready.")

---
## 6. Model 1 — Logistic Regression (Baseline)

In [ ]:
print("Training Logistic Regression...")

lr = LogisticRegression(
    labelCol=LABEL_COL, featuresCol="features",
    maxIter=100, regParam=0.01, elasticNetParam=0.0,
    weightCol="classWeight"
)

lr_model  = lr.fit(train_prep)
lr_preds  = lr_model.transform(test_prep)
lr_result = evaluate_model(lr_preds, "Logistic Regression")
results.append(lr_result)

lr_model.write().overwrite().save(f"{MODELS_DIR}/logistic_regression")
print("Model saved.")

---
## 7. Model 2 — Random Forest

In [ ]:
print("Training Random Forest...")

rf = RandomForestClassifier(
    labelCol=LABEL_COL, featuresCol="features",
    numTrees=100, maxDepth=10,
    minInstancesPerNode=10,
    featureSubsetStrategy="sqrt",
    seed=42, weightCol="classWeight"
)

rf_model  = rf.fit(train_prep)
rf_preds  = rf_model.transform(test_prep)
rf_result = evaluate_model(rf_preds, "Random Forest")
results.append(rf_result)

rf_model.write().overwrite().save(f"{MODELS_DIR}/random_forest")
print("Model saved.")

In [ ]:
# Feature importance — top 20
feature_names = (
    [f"{c}_ohe" for c in CATEGORICAL_COLS] +
    [f"{c}_imp" for c in NUMERIC_COLS]
)
importances = rf_model.featureImportances
feat_imp = pd.DataFrame({
    "feature":    feature_names[:len(feature_names)],
    "importance": [importances[i] for i in range(len(feature_names))]
}).sort_values("importance", ascending=False).head(20)

print("\nTop 20 Feature Importances (Random Forest):")
print(feat_imp.to_string(index=False))

---
## 8. Model 3 — Gradient Boosted Trees (GBT)

In [ ]:
print("Training Gradient Boosted Trees...")
print("(Slowest model — 5-15 mins expected)")

gbt = GBTClassifier(
    labelCol=LABEL_COL, featuresCol="features",
    maxIter=100, maxDepth=6,
    stepSize=0.1, subsamplingRate=0.8,
    minInstancesPerNode=10,
    seed=42, weightCol="classWeight"
)

gbt_model  = gbt.fit(train_prep)
gbt_preds  = gbt_model.transform(test_prep)
gbt_result = evaluate_model(gbt_preds, "Gradient Boosted Trees")
results.append(gbt_result)

gbt_model.write().overwrite().save(f"{MODELS_DIR}/gbt")
print("Model saved.")

---
## 9. Final Model Comparison

In [ ]:
comparison = pd.DataFrame(results)[
    ["model", "auc", "f1", "precision", "recall"]
].sort_values("auc", ascending=False)

print("\n" + "="*65)
print("  FINAL MODEL COMPARISON — Test Set (2022-2024)")
print("="*65)
print(comparison.to_string(index=False))
print("="*65)
best = comparison.iloc[0]
print(f"\n  Best model : {best['model']}")
print(f"  AUC-ROC    : {best['auc']:.4f}")
print(f"  F1 Score   : {best['f1']:.4f}")

---
## 10. Per-Airport Performance (Best Model)

In [ ]:
# Use GBT predictions — swap to rf_preds if RF won
best_preds = gbt_preds

(
    best_preds
    .groupBy("AIRPORT_CODE")
    .agg(
        F.count("*").alias("test_flights"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("actual_delay_pct"),
        F.round(F.avg("prediction") * 100, 1).alias("predicted_delay_pct"),
        F.round(
            F.sum(F.when(F.col(LABEL_COL) == F.col("prediction"), 1).otherwise(0))
            / F.count("*") * 100, 1
        ).alias("accuracy_pct")
    )
    .orderBy(F.desc("actual_delay_pct"))
    .show(25)
)

---
## 11. 2020 Performance Check

In [ ]:
# Check how well GBT handled 2020 vs normal years on train set
gbt_train_preds = gbt_model.transform(train_prep)

(
    gbt_train_preds
    .groupBy("is_covid_year")
    .agg(
        F.count("*").alias("flights"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("actual_delay_pct"),
        F.round(F.avg("prediction") * 100, 1).alias("predicted_delay_pct"),
        F.round(
            F.sum(F.when(F.col(LABEL_COL) == F.col("prediction"), 1).otherwise(0))
            / F.count("*") * 100, 1
        ).alias("accuracy_pct")
    )
    .orderBy("is_covid_year")
    .show()
)
print("is_covid_year=1 rows are 2020. Accuracy should remain high due to the flag.")

---
## 12. Summary

In [ ]:
print("="*60)
print("  PHASE 6 COMPLETE")
print("="*60)
print(f"  Train period  : 2015-2021 (incl. 2020 flagged)")
print(f"  Test period   : 2022-2024")
print(f"  Models saved  : {MODELS_DIR}")
print()
print("  Results:")
for r in results:
    print(f"    {r['model']:<30s}  AUC={r['auc']:.4f}  F1={r['f1']:.4f}")
print()
print("  Class imbalance handling:")
print(f"    Delayed class upweighted {weight_ratio:.1f}x")
print()
print("  2020 handling:")
print("    Included in training, flagged via is_covid_year=1")
print("    flight_volume_index captures traffic collapse")
print("="*60)

spark.stop()
print("Spark stopped.")